##PARTE 1: Consultas Básicas de Exploración

In [0]:
%sql
-- Familiarizarse con Databricks y ejecutar una primer query
-- ¿Cuántos viajes hay en total en el dataset de NYC Taxi Trips?
select count(*)
from samples.nyctaxi.trips;

In [0]:
%sql
-- Explorar estructura y Entender qué columnas tiene el dataset.
-- ¿Qué columnas tiene la tabla samples.nyctaxi.trips? ¿Qué tipos de datos tienen?
describe samples.nyctaxi.trips;

In [0]:
%sql
-- Encontrar valores extremos en los datos.
-- ¿Cuál es el viaje más largo (mayor distancia) en el dataset?
-- ¿Y el más corto (excluyendo ceros)?
select max(trip_distance), min(trip_distance)
from samples.nyctaxi.trips
where trip_distance > 0;


In [0]:
%sql
-- Estadísticas básicas. Calcular promedios y estadísticas descriptivas.
-- ¿Cuál es el promedio de tarifa (fare_amount) y distancia (trip_distance) de los viajes?
select avg(fare_amount) AS promoedio_tarifa, avg(trip_distance) AS promedio_distancia
from samples.nyctaxi.trips;

In [0]:
%sql
-- Viajes recientes (Ordenar y limitar resultados).
-- Muestra los 20 viajes más recientes con su fecha, distancia y tarifa.
select tpep_pickup_datetime AS fecha_y_hora_viaje, trip_distance as distancia, fare_amount as tarifa
from samples.nyctaxi.trips
order by tpep_pickup_datetime desc
limit 20;

##PARTE 2: Filtros y Condiciones

In [0]:
%sql
-- Aplicación de filtros simples con WHERE.
-- Muestra todos los viajes que tienen una distancia mayor a 10 millas.
-- Usando WHERE trip_distance > 10
select *
from samples.nyctaxi.trips
where trip_distance > 10;

In [0]:
%sql
-- Múltiples condiciones combinando condiciones con AND.
-- Encuentra viajes que tengan distancia > 5 millas
-- "Y" tarifa > $10. Pista: Usa WHERE ... AND ...
SELECT *
FROM samples.nyctaxi.trips
WHERE trip_distance > 5 AND fare_amount > 10;

In [0]:
%sql
-- Usando OR para condiciones alternativas.
-- Mostrar viajes que tengan distancia > a 20 millas "O" tarifa > a $100.
-- Usando WHERE ... OR ...
SELECT *
FROM samples.nyctaxi.trips
WHERE trip_distance > 20 OR fare_amount > 100;

In [0]:
%sql
-- Probando BETWEEN para Rango de valores
-- Encontrar viajes con tarifa entre $10 y $20, 
-- Usando WHERE fare_amount BETWEEN 10 AND 20
SELECT *
FROM samples.nyctaxi.trips
WHERE fare_amount BETWEEN 10 AND 20;

In [0]:
%sql
-- Filtrando por múltiples valores específicos.
-- Mostrando viajes que comenzaron en los pickup_zip = 10009, 10013 o 10014.
-- Usando WHERE pickup_zip IN (10009, 10013, 10014)
SELECT *
FROM samples.nyctaxi.trips
WHERE pickup_zip IN (10009, 10013, 10014);

##PARTE 3: Agregaciones

In [0]:
%sql
-- Conteo básico: Contar registros para saber ¿Cuántos viajes hay en total?
-- Usando SELECT COUNT(*) FROM ...
SELECT COUNT(*)
FROM samples.nyctaxi.trips;

In [0]:
%sql
-- Sumando valores numéricos: ¿Cuál es el total de tarifas y la distancia total recorrida por todos los viajes?
-- Usamos SUM() para ambas columnas con REDONDEO a 2 decimales.
SELECT
    round(SUM(fare_amount), 2) AS total_tarifa,
    round(SUM(trip_distance), 2) AS total_distancia
FROM samples.nyctaxi.trips;

In [0]:
%sql
-- Calculando promedios: ¿Cuál es la distancia promedio y la tarifa promedio de los viajes? (excluyendo valores cero)
-- Usamos AVG() con WHERE para filtrar ceros
SELECT
    round(AVG(trip_distance), 2) AS promedio_distancia,
    round(AVG(fare_amount), 2) AS promedio_tarifa
FROM samples.nyctaxi.trips
WHERE trip_distance > 0;

In [0]:
%sql
-- Encontrando valores extremos: ¿Cuál es la tarifa MIN, MAX, distancia MIN y MAX? (excluyendo valores cero)
-- Usamos MIN() y MAX() con filtros
SELECT
    round(MIN(fare_amount),2) AS tarifa_MIN,
    round(MAX(fare_amount),2) AS tarifa_MAX,
    MIN(trip_distance) AS distancia_MIN,
    MAX(trip_distance) AS distancia_MAX
FROM samples.nyctaxi.trips
WHERE trip_distance > 0 AND fare_amount > 0;

##PARTE 4: GROUP BY   ///   PARTE 5: HAVING

In [0]:
%sql
-- Agrupando datos por zona para calcular métricas por grupo.
-- ¿Cuántos viajes hay por zona de inicio (pickup_zip)?
-- ¿Cuál es el ingreso total y promedio por zona?
-- Ordenamos por ingreso total descendente.
-- Usamos GROUP BY pickup_zip, COUNT(*), SUM(fare_amount), AVG(fare_amount), ORDER BY
SELECT
    pickup_zip,
    COUNT(*) AS total_viajes,
    ROUND(SUM(fare_amount),2) AS total_ingreso,
    ROUND(AVG(fare_amount),2) AS promedio_ingreso
FROM samples.nyctaxi.trips
GROUP BY pickup_zip
ORDER BY total_ingreso DESC;


In [0]:
%sql
-- Buscando Viajes por día de la semana, agrupados por fecha con calculo de estadísticas.
-- ¿Cuántos viajes hay por día de la semana? ¿Cuál es la distancia promedio por día?
-- Usamos GROUP BY.
SELECT
    tpep_pickup_datetime,
    EXTRACT(DAYOFWEEK FROM tpep_pickup_datetime) AS dia_semana,
    COUNT(*) AS total_viajes,
    ROUND(AVG(trip_distance),2) AS promedio_distancia
FROM samples.nyctaxi.trips
GROUP BY tpep_pickup_datetime, dia_semana
ORDER BY total_viajes DESC;


In [0]:
%sql
-- Analizamos Viajes / hora buscando patrones horarios.
-- ¿Cuántos viajes hay por hora del día?
-- ¿Cuál es la tarifa promedio por hora? Ordenamos por hora.
-- Usamos EXTRACT(HOUR FROM ...) y GROUP BY
SELECT
    EXTRACT(HOUR FROM tpep_pickup_datetime) AS hora,
    COUNT(*) AS total_viajes,
    ROUND(AVG(fare_amount),2) AS promedio_tarifa
FROM samples.nyctaxi.trips
GROUP BY hora
ORDER BY hora;

In [0]:
%sql
-- Filtrando grupos grandes después de agrupar.
-- Muestrar solo las zonas que tienen + 1,000 viajes.
-- Usando GROUP BY con HAVING COUNT(*) > 1000
SELECT
    pickup_zip,
    COUNT(*) AS total_viajes
FROM samples.nyctaxi.trips
GROUP BY pickup_zip
HAVING total_viajes > 1000
ORDER BY total_viajes DESC;


In [0]:
%sql
-- Identificamos los días con mayor actividad.
-- ¿Qué días tuvieron más de 200 viajes?
-- Mostramos fecha, cantidad de viajes y tarifa promedio.
-- Usando GROUP BY, HAVING, ORDER BY y LIMIT
SELECT
    cast(tpep_pickup_datetime as date) AS fecha,
    COUNT(*) AS cantidad_de_viajes,
    ROUND(AVG(fare_amount),2) AS tarifa_promedio
FROM samples.nyctaxi.trips
GROUP BY
    cast(tpep_pickup_datetime as date)
HAVING cantidad_de_viajes > 200
ORDER BY cantidad_de_viajes DESC
LIMIT 10;

In [0]:
%sql
-- Identificando horas con mayor actividad.
-- ¿Qué horas tienen más de 1,000 viajes? Ordenamos por cantidad de viajes DESC.
-- Usamos GROUP BY, HAVING COUNT(*) > 1000
SELECT
    EXTRACT(HOUR FROM tpep_pickup_datetime) AS hora,
    COUNT(*) AS total_viajes
FROM samples.nyctaxi.trips
GROUP BY hora
HAVING total_viajes > 1000
ORDER BY total_viajes DESC;


##Parte 6: Análisis Combinado

In [0]:
%sql
-- Análisis completo por zona
-- Combinamos WHERE, GROUP BY y HAVING para responder:
-- Para zonas con + 1,000 viajes, muestra: zona, cantidad de viajes, tarifa máxima, promedio y mínima.
-- Solo mostraremos viajes con tarifa > $5 y distancia > 0.
-- Usaremos WHERE para filtrar registros, GROUP BY para agrupar, HAVING para filtrar grupos.
SELECT
    pickup_zip AS zona,
    COUNT(*) AS cantidad_de_viajes,
    ROUND(MAX(fare_amount),2) AS tarifa_maxima,
    ROUND(AVG(fare_amount),2) AS tarifa_promedio,
    ROUND(MIN(fare_amount),2) AS tarifa_minima
FROM samples.nyctaxi.trips
WHERE trip_distance > 0 AND fare_amount > 5
GROUP BY pickup_zip
HAVING cantidad_de_viajes > 1000
ORDER BY cantidad_de_viajes DESC;

In [0]:
%sql
-- Categorizando viajes por distancia. Usaremos CASE WHEN para crear categorías.
-- Categorizar los viajes por distancia:
-- Corto: < 2 millas
-- Medio: 2-5 millas
-- Largo: 5-10 millas
-- Muy largo: > 10 millas
-- Agrupamos por categoría y mostramos: categoría, cantidad de viajes, tarifa promedio, distancia promedio. Solo categorías con más de 100 viajes.
-- Usaremos CASE WHEN en el SELECT y también en el GROUP BY.
SELECT
    CASE WHEN trip_distance < 2 THEN 'Corto'
         WHEN trip_distance >= 2 AND trip_distance < 5 THEN 'Medio'
         WHEN trip_distance >= 5 AND trip_distance < 10 THEN 'Largo'
         WHEN trip_distance >= 10 THEN 'Muy largo'
    END AS categoria,
    COUNT(*) AS cantidad_de_viajes,
    ROUND(AVG(fare_amount),2) AS tarifa_promedio,
    ROUND(AVG(trip_distance),2) AS distancia_promedio
FROM
    samples.nyctaxi.trips
GROUP BY
    categoria
HAVING
    cantidad_de_viajes > 100
ORDER BY
    cantidad_de_viajes DESC;

In [0]:
%sql
-- Buscando el "Top Ten" de viajes.
-- Mostraremos los 10 viajes más largos (mayor distancia) con su fecha, distancia y tarifa.
-- Usaremos ORDER BY trip_distance DESC LIMIT 10
SELECT
    tpep_pickup_datetime,
    trip_distance,
    fare_amount
FROM
    samples.nyctaxi.trips
ORDER BY
    trip_distance DESC
LIMIT 10;


In [0]:
%sql
-- el resultado anterior nos ha devuelto un viaje con fare_amount = 0, lo cual no es correcto o no los queremos en el top ten por esa razón (pudo haber sido una "promoción").
-- Reescribimos la query para excluírlos.
SELECT
    tpep_pickup_datetime,
    trip_distance,
    fare_amount
FROM
    samples.nyctaxi.trips
WHERE
    fare_amount > 0
    AND fare_amount IS NOT NULL
    AND trip_distance IS NOT NULL
ORDER BY
    trip_distance DESC
LIMIT 10;